In [8]:
from utils.logger import logger
from utils.config import config
from pipeline.pdf_parser import GrobidPDFParser
from pipeline.sentence_extractor import extract_sentences
from pipeline.reference_resolver import ReferenceResolver
from pipeline.classifier import GeminiClassifier


In [2]:
logger.info("Starting the application...")

logger.info("Parsing the PDF and extracting information...")

parser = GrobidPDFParser(pdf_path="../papers/BERT.pdf")
parsed_paper = parser.parse()

logger.info("Successfully parsed the PDF. Extracted information:")
logger.info(f"Title: {parsed_paper.title}")
logger.info(f"Abstract: {parsed_paper.abstract}")

2026-05-09 16:08:57,653 - missing_citations - INFO - Starting the application...
2026-05-09 16:08:57,654 - missing_citations - INFO - Parsing the PDF and extracting information...
2026-05-09 16:09:12,745 - missing_citations - INFO - Successfully parsed the PDF. Extracted information:
2026-05-09 16:09:12,746 - missing_citations - INFO - Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
2026-05-09 16:09:12,747 - missing_citations - INFO - Abstract: We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a;[CITE:b36], BERT is designed to pretrain deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be finetuned with just one additional output layer to create state-of-the-art models for a wide ra

In [3]:
logger.info("Extracting sentences from the parsed paper...")
sentences = extract_sentences(parsed_paper)
logger.info(f"Extracted {len(sentences)} sentences from the paper.")


2026-05-09 16:09:12,757 - missing_citations - INFO - Extracting sentences from the parsed paper...
2026-05-09 16:09:15,842 - missing_citations - INFO - Extracted 287 sentences from the paper.


In [4]:
logger.info("Resolving references in the paper...")
resolver = ReferenceResolver()
resolved_references = []

for ref in parsed_paper.references:
    resolved = resolver.resolve(ref)
    resolved_references.append(resolved)

logger.info("Resolved references:")
for ref, resolved in zip(parsed_paper.references, resolved_references):
    logger.info(f"Original: {ref}")
    logger.info(f"Resolved: {resolved}")
    print("---")

logger.info(f"Stats: {resolver.stats}")


2026-05-09 16:09:15,855 - missing_citations - INFO - Resolving references in the paper...
2026-05-09 16:10:05,762 - missing_citations - INFO - Resolved references:
2026-05-09 16:10:05,764 - missing_citations - INFO - Original: Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.
2026-05-09 16:10:05,765 - missing_citations - INFO - Resolved: ResolvedReference(raw_reference='Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.', resolved_paper_id='W2880875857', openalex_id='W2880875857', title='Contextual String Embeddings for Sequence Labeling', doi=None, method='openalex', confidence=1.0, unresolved_reason=None)
---
2026-05-09 16:10:05,766 - missing_citations - INFO - Original: Rami Al-R

In [5]:
if resolver.stats.get("openalex_external", -1) not in [-1, 0]:
    from sentence_transformers import SentenceTransformer
    from fastembed import SparseTextEmbedding
    from database.qdrant import create_qdrant_client
    from utils.config import config

    logger.info("Initializing models and Qdrant client...")

    # Initialize Qdrant client
    qdrant_client = create_qdrant_client(config.QDRANT_URL)
    logger.info(f"Connected to Qdrant at {config.QDRANT_URL}")

    # Load dense model
    logger.info(f"Loading dense model: {config.DENSE_MODEL}...")
    dense_model = SentenceTransformer(config.DENSE_MODEL)
    logger.info("Dense model loaded")

    # Load sparse model
    logger.info(f"Loading sparse model: {config.SPARSE_MODEL}...")
    sparse_model = SparseTextEmbedding(model_name=config.SPARSE_MODEL)
    logger.info("Sparse model loaded")

In [6]:
from ingest_missing import ingest_openalex_papers
from indexer import EmbeddingIndex

# 1. Collect the IDs of all papers that were found externally
missing_ids = [
    ref.openalex_id 
    for ref in resolved_references 
    if ref.method == "openalex_external" and ref.openalex_id
]

if missing_ids:
    # 2. You will need to pass your initialized EmbeddingIndex. 
    # (Assuming you already have your qdrant_client, dense_model, etc. initialized)
    embedding_idx = EmbeddingIndex(
        qdrant_client=qdrant_client,
        dense_model=dense_model,
        sparse_model=sparse_model
    )
    
    # 3. Fetch, insert to Postgres, embed, and insert to Qdrant!
    inserted_count = ingest_openalex_papers(missing_ids, embedding_idx)
    print(f"Successfully ingested {inserted_count} missing papers into the local corpus.")


c:\Users\sampe\OneDrive\Desktop\facultate\licenta\missing-citations-identifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
"""
STAGE 4A - Citation Worthiness Classification with GEMINI CLASSIFIER
"""

logger.info("Classifying sentences for citation worthiness using Gemini Classifier...")
gemini_classifier = GeminiClassifier(model=config.CLASSIFIER_BACKUP[0], batch_size=31)

classified_sentences = gemini_classifier.classify_sentences(sentences[:30], parsed_paper.title, parsed_paper.abstract)

logger.info("Classification results:")
for i, sentence in enumerate(classified_sentences):
    logger.info(f"Classification {i}: {sentence.__dict__}")
    print("---")

2026-05-09 16:10:42,270 - missing_citations - INFO - Classifying sentences for citation worthiness using Gemini Classifier...
2026-05-09 16:10:44,602 - missing_citations - INFO - Sending batch 1/1 to Gemini...
2026-05-09 16:10:52,406 - missing_citations - INFO - Classification results:
2026-05-09 16:10:52,407 - missing_citations - INFO - Classification 0: {'text': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers.', 'section': 'Abstract', 'position_in_section': 0.0, 'has_citation': False, 'citation_intent': <CitationIntent.OTHER: 'OTHER'>, 'retrieval_text': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers.', 'previous_sentence': None, 'next_sentence': 'Unlike recent language representation models (Peters et al., 2018a;[CITE:b36], BERT is designed to pre-train deep bidirectional representations from unlabeled text by